# Reproduce E2E — SQLite에서 실험 로드 후 재현

`optuna_{USER}.db`에 저장된 study를 읽어 **HPO 없이** 저장된 `best_params`(또는 지정 trial)로 학습 1회만 돌려 동일한 예측을 재현한다.

- `rerun_best_trial_with_pp`가 전처리(cleaning/outlier) + CLF + 집계 + REG를 한 번에 처리한다.
- 동일 SEED + 동일 파라미터이지만, 실행 환경(GPU/CPU, 라이브러리 버전)에 따라 미세한 차이가 날 수 있다.
- `target_transform`이 설정된 실험은 Y를 변환 후 학습 → 예측 역변환을 자동 수행한다.

## 1. 환경 설정 및 데이터 로드

In [1]:
# ============================================================
# 환경 설정 + 데이터 로드
# ============================================================
import os, sys

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/e2e_hpo.py'):
        os.system('gdown 1Vrn5LBl611rWbag7d09LZH68_lfpu6wP -O /content/modules.zip')
        os.makedirs('/content/project/3_modeling/modules', exist_ok=True)
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling/modules')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

# --- 기본 라이브러리 ---
import numpy as np
import pandas as pd
import optuna
import warnings
warnings.filterwarnings('ignore')

# --- 프로젝트 유틸 ---
from utils.config import PROJECT_ROOT, OUTPUT_DIR, SEED, TARGET_COL, KEY_COL, POSITION_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import evaluate, rmse
from utils.experiment import download_from_drive

# --- 전처리 모듈 (rerun_best_trial_with_pp 내부에서 import하지만, path 설정은 필요) ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '2_preprocessing'))

# --- 3_modeling 모듈 ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '3_modeling'))
from modules.e2e_hpo import rerun_best_trial_with_pp

# --- 데이터 로드 ---
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

setup 완료
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749
Feature 수: 1087
Die 수: train=104,988, val=34,996, test=34,996


## 2. 실험 설정 로드 (SQLite → 파라미터 주입)

In [2]:
# ============================================================
# 실험 설정 로드 — SQLite에서 study + trial 파라미터 읽어와서 주입
# ============================================================

# ── 사용자 지정 ──
EXP_ID       = '60-904-001'   # 재현할 실험번호 (= study_name)
USER         = 'jh'          # DB 파일 소유자
TRIAL_NUMBER = None          # None → best_trial 자동 선택, 정수 → 해당 trial 지정
EVAL_TEST    = True          # True: test RMSE 계산 (test peeking), False: val만

# ── Google Drive 파일 ID (Colab에서 .db 최신본 다운로드용) ──
CSV_GDRIVE_ID = ''           # 공용 experiments.csv (빈 문자열이면 스킵)
DB_GDRIVE_ID  = ''           # 본인 optuna_{USER}.db (빈 문자열이면 스킵)

# ── DB 경로 결정 ──
DB_PATH = os.path.join(OUTPUT_DIR, 'experiments', f'optuna_merged.db')

# Colab이면 Drive에서 다운로드
if DB_GDRIVE_ID:
    download_from_drive(csv_gdrive_id=CSV_GDRIVE_ID, db_gdrive_id=DB_GDRIVE_ID, db_path=DB_PATH)

assert os.path.exists(DB_PATH), f"DB 파일이 없습니다: {DB_PATH}"

# ── SQLite에서 Study 로드 ──
study = optuna.load_study(
    study_name=EXP_ID,
    storage=f"sqlite:///{DB_PATH}",
)
print(f'Study "{EXP_ID}" 로드 완료: {len(study.trials)}개 trial')

# ── Trial 선택 ──
if TRIAL_NUMBER is not None:
    matched = [t for t in study.trials if t.number == TRIAL_NUMBER]
    if not matched:
        raise ValueError(
            f"Trial #{TRIAL_NUMBER}이 study '{EXP_ID}'에 없습니다. "
            f"사용 가능: {[t.number for t in study.trials]}"
        )
    trial = matched[0]
    print(f'지정 trial #{trial.number} 선택 (value={trial.value:.8f})')
else:
    trial = study.best_trial
    print(f'Best trial #{trial.number} 선택 (value={trial.value:.8f})')

best_params = trial.params

# ── study.user_attrs에서 메타데이터 복원 ──
ua = study.user_attrs
pipeline_config = ua['pipeline_config']
e2e_params      = ua['e2e_params']
rerun_params    = ua['rerun_params']
sampling_params = ua['sampling_params']
LABEL_COL       = ua.get('label_col', 'label_bin')
EXCLUDE_COLS    = ua.get('exclude_cols', []) or []
TARGET_TRANSFORM = ua.get('target_transform', 'none')

# ── 저장된 최종 결과 (비교용) ──
final_method       = ua.get('final_method')
expected_val_rmse  = ua.get('final_val_rmse')
expected_test_rmse = ua.get('final_test_rmse')

# ── trial.user_attrs 참고 정보 ──
trial_ua = trial.user_attrs
print(f'\n── Trial #{trial.number} 정보 ──')
print(f'  train_rmse  : {trial_ua.get("train_rmse", "N/A")}')
print(f'  val_rmse    : {trial_ua.get("val_rmse", "N/A")}')
print(f'  n_feat_clean: {trial_ua.get("n_feat_clean", "N/A")}')
print(f'  agg_funcs   : {trial_ua.get("agg_funcs", "N/A")}')

# ── 설정 출력 ──
print(f'\n재현 대상       : {EXP_ID} (trial #{trial.number})')
print(f'final_method    : {final_method}')
print(f'CLF / REG       : {e2e_params["clf_model"]} / {e2e_params["reg_model"]}')
print(f'Rerun mode      : {rerun_params["mode"]} (n_folds={rerun_params.get("n_folds")})')
print(f'타깃 변환       : {TARGET_TRANSFORM}')
print(f'샘플링          : {"ON" if sampling_params["use_sampling"] else "OFF"}')
print(f'EVAL_TEST       : {EVAL_TEST}')
print(f'─── 저장된 결과 ───')
print(f'final_val_rmse  : {expected_val_rmse}')
print(f'final_test_rmse : {expected_test_rmse}')
print(f'\n── Pipeline Config ──')
for k, v in pipeline_config.items():
    print(f'  {k}: {v}')

Study "60-904-001" 로드 완료: 330개 trial
Best trial #230 선택 (value=0.00115125)

── Trial #230 정보 ──
  train_rmse  : 0.0011512494875316592
  val_rmse    : 0.0011543007745008107
  n_feat_clean: 740
  agg_funcs   : ['mean', 'std']

재현 대상       : 60-904-001 (trial #230)
final_method    : None
CLF / REG       : lgbm / lgbm
Rerun mode      : kfold (n_folds=5)
타깃 변환       : yeo-johnson
샘플링          : OFF
EVAL_TEST       : True
─── 저장된 결과 ───
final_val_rmse  : None
final_test_rmse : None

── Pipeline Config ──
  input_level: die
  run_clf: False
  clf_output: proba
  clf_filter: True
  clf_optuna: True
  run_fs: False
  fs_optuna: False
  reg_level: position
  reg_optuna: True
  zero_clip: False


## 3. 타깃 변환 + 저장된 best_params로 학습 1회 (HPO 스킵)

In [3]:
# ============================================================
# 타깃 변환 → HPO 스킵 → 저장된 best_params로 학습 1회
# ============================================================

# ── 타깃 변환 (baseline.ipynb과 동일 로직) ──
target_transformer = None
ys_input = {k: v.copy() for k, v in ys.items()}

if TARGET_TRANSFORM == 'log1p':
    for _split, _df in ys_input.items():
        if TARGET_COL in _df.columns:
            _df[TARGET_COL] = np.log1p(_df[TARGET_COL])
    _train_y = ys_input['train'][TARGET_COL]
    print(f'[log1p] target 변환 적용: '
          f'min={_train_y.min():.6f}, max={_train_y.max():.6f}')

elif TARGET_TRANSFORM == 'yeo-johnson':
    from sklearn.preprocessing import PowerTransformer
    target_transformer = PowerTransformer(method='yeo-johnson', standardize=False)
    _train_y_raw = ys_input['train'][TARGET_COL].values.reshape(-1, 1)
    target_transformer.fit(_train_y_raw)
    for _split, _df in ys_input.items():
        if TARGET_COL in _df.columns:
            _df[TARGET_COL] = target_transformer.transform(
                _df[TARGET_COL].values.reshape(-1, 1)
            ).ravel()
    _train_y = ys_input['train'][TARGET_COL]
    print(f'[yeo-johnson] target 변환 적용: '
          f'min={_train_y.min():.6f}, max={_train_y.max():.6f}')
else:
    print('[none] target 변환 없음 — 원본 스케일 사용')

# ── 역변환 함수 정의 ──
def _inv_transform(arr):
    if TARGET_TRANSFORM == 'log1p':
        return np.expm1(arr)
    elif TARGET_TRANSFORM == 'yeo-johnson':
        return target_transformer.inverse_transform(
            arr.reshape(-1, 1)
        ).ravel()
    return arr  # 'none'

# ── rerun_best_trial_with_pp (전처리 + CLF + 집계 + REG 한 번에) ──
final = rerun_best_trial_with_pp(
    xs=xs,
    xs_dict=xs_dict,
    ys=ys_input,
    feat_cols=feat_cols,
    best_params=best_params,
    pipeline_config=pipeline_config,
    clf_model=e2e_params['clf_model'],
    reg_model=e2e_params['reg_model'],
    mode=rerun_params['mode'],
    n_folds=rerun_params.get('n_folds', 5),
    es_holdout=rerun_params.get('es_holdout', 0.1),
    clf_early_stop=rerun_params.get('clf_early_stop', 100),
    reg_early_stop=rerun_params.get('reg_early_stop', 100),
    label_col=LABEL_COL,
    imbalance_method=e2e_params.get('imbalance_method', 'scale_pos_weight'),
    top_k_fixed=e2e_params.get('top_k_fixed', 200),
    clf_fixed=e2e_params.get('clf_fixed'),
    reg_fixed=e2e_params.get('reg_fixed'),
    use_sampling=sampling_params.get('use_sampling', False),
    sample_frac=sampling_params.get('sample_frac', 1.0),
    exclude_cols=EXCLUDE_COLS,
)

# ── Val 평가 ──
# 원본 스케일의 Y (변환 전)
y_val_raw = ys['validation'][TARGET_COL].values
# rerun 결과는 변환된 스케일 → 역변환
val_pred_raw = _inv_transform(final['val_pred'])

repro_val_rmse = float(rmse(y_val_raw, val_pred_raw))
print(f'\n[Reproduce {EXP_ID} (val)] RMSE = {repro_val_rmse:.8f}  '
      f'(n={len(y_val_raw):,}, zero={(y_val_raw == 0).sum():,})')
evaluate(y_val_raw, val_pred_raw, label=f'Reproduce {EXP_ID} (val)')

# ── Test 평가 ──
if EVAL_TEST:
    y_test_raw = ys['test'][TARGET_COL].values
    test_pred_raw = _inv_transform(final['test_pred'])
    repro_test_rmse = float(rmse(y_test_raw, test_pred_raw))
    print(f'[Reproduce {EXP_ID} (test)] RMSE = {repro_test_rmse:.8f}  '
          f'(n={len(y_test_raw):,}, zero={(y_test_raw == 0).sum():,})')
    evaluate(y_test_raw, test_pred_raw, label=f'Reproduce {EXP_ID} (test)')
else:
    repro_test_rmse = None
    print('[test] EVAL_TEST=False → test RMSE 계산 생략')

[yeo-johnson] target 변환 적용: min=-0.000000, max=0.009056
Rerun preprocessing: cleaning=12 args, outlier method=winsorize, agg_funcs=['mean', 'std']
클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104988, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 951개
    컬럼: 977 → 951 (26개 제거)
    DataFrame: (104988, 955)

[고상관 제거] threshold=0.98, keep_by=std (std)
  제거: 137개, 잔여: 814개
    컬럼: 951 → 814 (137개 제거)
    DataFrame: (104988, 818)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 20%)
[공간 보간 imputation] 총 결측: 678,896
  train-only 모드: train 104,988 / 전체 174,980 행
  1단계 (공간 보간, dist<=3.0): 118,253개 채움 → 잔여: 560,643
  2단계 (lot 평균, train 기준): 479,447개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 678,896 → 공간(118,253) → lot(479,447) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.97, keep_by=target_corr (|target 상관| 우선, 동률

## 4. 재현 검증 — 저장된 RMSE vs 재현 RMSE

In [4]:
# ============================================================
# 재현 검증 — 저장된 RMSE 와 재현 RMSE 비교
# ============================================================
# 같은 파라미터 + 같은 SEED 이지만 환경(GPU/CPU/라이브러리 버전) 차이로
# bit-identical 이 아닐 수 있다. 아래 3단계로 판단한다.

TOL_EXACT = 1e-8
TOL_CLOSE = 1e-4

def _report(name, expected, reproduced):
    print(f'── {name} ──')
    if expected is None:
        print(f'  기대값: N/A (study에 기록 없음)')
        print(f'  재현값: {reproduced:.8f}' if reproduced is not None else '  재현값: N/A')
        return
    if reproduced is None:
        print(f'  기대값: {expected:.8f}')
        print(f'  재현값: N/A (EVAL_TEST=False)')
        return
    diff = reproduced - expected
    rel = abs(diff) / max(abs(expected), 1e-12)
    print(f'  기대값: {expected:.8f}')
    print(f'  재현값: {reproduced:.8f}')
    print(f'  차이  : {diff:+.2e}  (상대 {rel:.2e})')
    if abs(diff) < TOL_EXACT:
        print(f'  ✓ 완전 일치 (< {TOL_EXACT:.0e})')
    elif abs(diff) < TOL_CLOSE:
        print(f'  ○ 근사 일치 (< {TOL_CLOSE:.0e}) — 결정성 차이 범위')
    else:
        print(f'  ⚠ 불일치 — 환경/버전/GPU-CPU 차이 의심')

_report('Val RMSE', expected_val_rmse, repro_val_rmse)
if EVAL_TEST:
    _report('Test RMSE', expected_test_rmse, repro_test_rmse)

── Val RMSE ──
  기대값: N/A (study에 기록 없음)
  재현값: 0.00582319
── Test RMSE ──
  기대값: N/A (study에 기록 없음)
  재현값: 0.00850542
